**Setup**

Import the required libraries and load the chatbot conversation dataset.

* Dataset: bitext/customer-support-llm-chatbot-training-dataset (Modified)
* Original: https://huggingface.co/datasets/bitext/customer-support-llm-chatbot-training-dataset

MODIFICATIONS (Educational):
- Added synthetic PII columns and injected PII into text
- Introduced data quality issues
- Purpose: Simulate realistic data preprocessing challenges

In [1]:
import re
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

# Load the chatbot conversation dataset
df = pd.read_csv('./data/customer_support_clean.csv')

**Example**

You'll work with production conversation logs from your AI chatbot. Each row represents one conversation with metrics like response time, cost, and quality indicators. Explore a sample conversation:


In [2]:
sample = df.iloc[100]
print("=== Sample Conversation ===")
#print(f"Conversation ID: {sample['conversation_id']}")
print(f"Model used: {sample['model_used']}")
print(f"Latency: {sample['latency_seconds']:.2f}s")
print(f"Tokens used: {sample['tokens_used']}")
print(f"Cost: ${sample['cost']:.4f}")
print(f"Resolved: {sample['resolved']}")
print(f"CSAT: {sample['csat_score']}/5")
print(f"Hallucinated: {sample['hallucinated']}")
print(f"Columns: {', '.join(df.columns)}")

=== Sample Conversation ===
Model used: gpt-4
Latency: 2.78s
Tokens used: 194
Cost: $0.0058
Resolved: True
CSAT: 3.1/5
Hallucinated: False
Columns: customer_id_hashed, customer_type, zipcode, week_number, flags, instruction, response, category, intent, model_tier, model_used, tokens_used, cost, latency_seconds, conversation_turns, resolved, escalated_to_human, csat_score, hallucinated, error_occurred


#### Detect Jailbreak Attempts in Real Conversations

Some users attempt to trick our chatbot with jailbreak prompts, such as "Ignore your previous instructions" or "Pretend you're not an AI." Our security team needs to identify these attempts to enhance our defenses and understand how individuals attempt to compromise our system.

1. Update the function `detect_jailbreak(text)` that checks if text contains any jailbreak patterns, and if so, returns `True` and the jailbreak pattern found; if no jailbreak pattern is detected, the function should return `False` and `None`
2. Apply this function to the `instruction` column to flag suspicious conversations
3. Count how many conversations contain jailbreak attempts
4. Print the counts for each pattern detected

**Jailbreak patterns to detect:**
- "ignore previous instructions"
- "disregard your programming" 
- "act as if you are"
- "pretend you are not an ai"
- "forget your rules"
- "system prompt"
- "new instructions"

In [3]:
JAILBREAK_PATTERNS = [
    "ignore previous instructions",
    "disregard your programming",
    "act as if you are",
    "pretend you are not an ai",
    "forget your rules",
    "system prompt",
    "new instructions"
]

def detect_jailbreak(text):
    """
    Detect if text contains jailbreak attempt patterns.
    
    Args:
        text (str): Text to check for jailbreak patterns
        
    Returns:
        tuple: (has_jailbreak: bool, matched_pattern: str or None)
    """
    if pd.isna(text):
        return False, None
    
    text_lower = str(text).lower()
    
    for pattern in JAILBREAK_PATTERNS:
        if pattern in text_lower:
            return True, pattern #update return values
    
    return False, None

# Apply detection to all instructions
df[['has_jailbreak', 'jailbreak_pattern']] = df['instruction'].apply(lambda x: pd.Series(detect_jailbreak(x)))

# Count jailbreak attempts
jailbreak_count = df['has_jailbreak'].sum()
jailbreak_pct = (jailbreak_count / len(df['instruction'])) * 100

print("=== Jailbreak Detection Results ===")
print(f"\nTotal conversations: {len(df):,}")
print(f"Jailbreak attempts detected: {jailbreak_count:,} ({jailbreak_pct:.2f}%)")

# Pattern frequency analysis
if jailbreak_count > 0:
    pattern_counts = df[df['has_jailbreak']]['jailbreak_pattern'].value_counts()
    print(f"\nMost common jailbreak patterns:")
    for pattern, count in pattern_counts.head().items():
        print(f"  • '{pattern}': {count} times")


=== Jailbreak Detection Results ===

Total conversations: 26,872
Jailbreak attempts detected: 0 (0.00%)


#### Identify PII Leakage in Bot Responses

Our chatbot should never include real customer PII (such as emails, phone numbers, and credit card information) in its responses. But due to a bug, the bot occasionally repeats sensitive information. You need to scan all responses to identify where PII is being leaked.

1. Update the functions to detect PII patterns in text:
   - `contains_email(text)` - detects email addresses
   - `contains_phone(text)` - detects phone numbers (format: XXX-XXX-XXXX or similar)
   - `contains_credit_card(text)` - detects credit card patterns (16 digits with optional dashes/spaces)
2. Apply these functions to the `response` column to flag PII leakage
3. Count how many responses contain each type of PII
4. Calculate the PII leakage rate (% of responses with PII)
5. Print examples of responses that leaked PII

**Hint:** Use regular expressions (regex) to match patterns:
```python
import re
EMAIL_PATTERN = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
has_email = bool(re.search(EMAIL_PATTERN, text))
```

In [4]:
# PII detection patterns
EMAIL_PATTERN = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
PHONE_PATTERN = r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'
CC_PATTERN = r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'

def contains_email(text):
    """Check if text contains an email address."""
    if pd.isna(text):
        return False
    return bool(re.search(EMAIL_PATTERN, str(text)))

def contains_phone(text):
    """Check if text contains a phone number."""
    if pd.isna(text):
        return False
    return bool(re.search(PHONE_PATTERN, str(text)))

def contains_credit_card(text):
    """Check if text contains a credit card number."""
    if pd.isna(text):
        return False
    return bool(re.search(CC_PATTERN, str(text)))

# Apply PII detection to all responses
df['leaked_email'] = df['response'].apply(contains_email)
df['leaked_phone'] = df['response'].apply(contains_phone)
df['leaked_cc'] = df['response'].apply(contains_credit_card)

# Create overall PII leakage flag
df['leaked_pii'] = df[['leaked_email', 'leaked_phone', 'leaked_cc']].any(axis=1)

# Count PII leakage
total_responses = len(df['response'])
email_leaks = df['leaked_email'].sum()
phone_leaks = df['leaked_phone'].sum()
cc_leaks = df['leaked_cc'].sum()
total_leaks = df['leaked_pii'].sum()

print("=== PII Leakage Detection Results ===")
print(f"\nTotal responses analyzed: {total_responses:,}")
print(f"Responses with PII leakage: {total_leaks:,} ({(total_leaks/total_responses)*100:.2f}%)")
print(f"\nBreakdown by PII type:")
print(f"  • Email addresses: {email_leaks:,} ({(email_leaks/total_responses)*100:.2f}%)")
print(f"  • Phone numbers: {phone_leaks:,} ({(phone_leaks/total_responses)*100:.2f}%)")
print(f"  • Credit cards: {cc_leaks:,} ({(cc_leaks/total_responses)*100:.2f}%)")

    
print(f"\n=== Example PII Leakage Cases ===")

leak_example = df[df['leaked_pii']==True].iloc[0]
print(f"\nPII Leak Example:")
print(f"Category: {leak_example['category']}")
print(f"Response: {leak_example['response']}")

=== PII Leakage Detection Results ===

Total responses analyzed: 26,872
Responses with PII leakage: 55 (0.20%)

Breakdown by PII type:
  • Email addresses: 55 (0.20%)
  • Phone numbers: 1 (0.00%)
  • Credit cards: 0 (0.00%)

=== Example PII Leakage Cases ===

PII Leak Example:
Category: CONTACT
Response: Thank you for reaching out! I completely understand your need for an email to contact our customer support team. We do have an email address dedicated to customer inquiries, and I'd be happy to provide it for you. Please send your question or concern to our customer support email at support@example.com. Our team will promptly assist you and ensure your concerns are addressed. Is there anything else I can assist you with?


#### Analyze Abuse Patterns and Cost Impact

We think some users are abusing our chatbot—sending too many requests, attempting jailbreaks, or causing costly errors. We need to identify these users and calculate the financial impact to justify adding rate limits.

1. Identify users with suspicious behavior patterns:
   - High request volume (>50 conversations)
   - Excessive token usage (>300 tokens per conversation on average)
2. Calculate the total cost impact of abusive users vs. normal users

In [5]:
# Analyze user behavior patterns
user_stats = df.groupby('customer_id_hashed').agg(
    total_requests=pd.NamedAgg(column='response', aggfunc='count'),
    total_tokens=pd.NamedAgg(column='tokens_used', aggfunc='mean'),
    total_cost=pd.NamedAgg(column='cost', aggfunc='sum')
).reset_index()

# Define abuse criteria
high_volume = user_stats['total_requests'] > 50
excessive_tokens = user_stats['total_tokens'] > 300

# Flag abusive users
user_stats['is_abusive'] = high_volume | excessive_tokens

# Separate abusive and normal users
abusive_users = user_stats[user_stats['is_abusive']]
normal_users = user_stats[~user_stats['is_abusive']]

print("=== Abuse Pattern Analysis ===")
print(f"\nTotal unique users: {len(user_stats):,}")
print(f"Abusive users identified: {len(abusive_users):,} ({(len(abusive_users)/len(user_stats))*100:.1f}%)")
print(f"Normal users: {len(normal_users):,} ({(len(normal_users)/len(user_stats))*100:.1f}%)")

# Cost impact analysis
print(f"\n=== Cost Impact Analysis ===")

total_cost = user_stats['total_cost'].sum()
abusive_cost = abusive_users['total_cost'].sum()
normal_cost = normal_users['total_cost'].sum()
abusive_cost_pct = (abusive_cost / total_cost) * 100

print(f"\nTotal API costs: ${total_cost:,.2f}")
print(f"Cost from abusive users: ${abusive_cost:,.2f} ({abusive_cost_pct:.1f}% of total)")
print(f"Cost from normal users: ${normal_cost:,.2f} ({(normal_cost/total_cost)*100:.1f}% of total)")


=== Abuse Pattern Analysis ===

Total unique users: 23,182
Abusive users identified: 15 (0.1%)
Normal users: 23,167 (99.9%)

=== Cost Impact Analysis ===

Total API costs: $91.16
Cost from abusive users: $0.11 (0.1% of total)
Cost from normal users: $91.04 (99.9% of total)
